# NLP extractor: experiments and final pipeline

From the first English biomedical NER baseline to the selected multilingual extractor. Each trial has its question, recorded result and takeaway. The last sections run the current code.

**Run:** Python 3.12 → `pip install -r requirements.txt` → restart kernel → **Run All**. Keep `src/`, `artifacts/` and `models/` beside this notebook. Default execution is offline, without fitting or downloads.

**Contents:** [Setup](#setup) · [History](#experiment-history) · [NER baseline](#1-english-biomedical-ner-baseline) · [Concept retrieval](#2-multilingual-concept-retrieval) · [Full-fact trials](#3-full-fact-extraction-trials) · [Tokenizers](#4-tokenizer-investigation) · [Current extractor](#5-selected-extractor-live-examples) · [Reference checks](#6-current-reference-checks) · [Conversations](#7-multi-turn-behavior) · [Conclusions](#8-conclusions-and-next-work)

Historical results and current runs are labelled separately. Synthetic references and weak annotations do not measure clinical accuracy.

## Setup

Load the local implementation and verify source/model files. Setup details live in [notebook_support.py](src/notebook_support.py).

In [1]:
from pathlib import Path
import sys, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd().resolve()
if not (PACKAGE / 'src/notebook_support.py').is_file():
    PACKAGE = PACKAGE / 'notebook-release'
sys.path.insert(0, str(PACKAGE / 'src'))
from notebook_support import initialize, read_json, facts_frame, prediction_row

ARTIFACTS = initialize(PACKAGE)
WORK = PACKAGE / '.notebook_runtime'
RESEARCH = ARTIFACTS / 'multilingual'
from triage_pipeline import extract_info, to_features, process_turn, load_state_model, predict_from_state
pd.set_option('display.max_colwidth', 65)
pd.set_option('display.precision', 4)
print('Ready: source and model hashes verified.')

Ready: source and model hashes verified.


## Experiment history

| Stage | Approach | Measurement | Decision |
|---|---|---|---|
| Initial English model | Biomedical NER | 100-question entity/leakage audit | Starting point; no gold-span accuracy |
| Multilingual expansion | Aligned phrases vs MiniLM sentence retrieval | Concept matches on development/stress examples | Keep aligned phrases |
| Comparable extraction | Dictionaries, MiniLM spans, GLiNER INT8/FP32, hybrids | Exact entities, complete facts, critical recall and latency | Dictionaries remain default |
| Language coverage | TunBERT, XLM-R, vioBERT, GLiNER tokenizers | Fragmentation and unknown tokens | Encoding coverage alone cannot select an extractor |
| Integration | Clinical context, linked answers and rules | Development references and software examples | Experimental; gaps retained |

## 1. English biomedical NER baseline

**Question:** Can an English pretrained biomedical NER model supply a first extractor?

**Historical result:** `d4data/biomedical-ner-all` processed 100 sampled questions, producing 1,478 predicted mentions. No manually annotated span gold was available. The entity audit flagged possible answer leakage; it did not establish extraction accuracy.

**Takeaway:** Useful as a starting point, with clinical span/context references still needed.

[Original baseline code and complete measured notes](research/01_medical_nlp_baseline.py). The original environment and downloads are not rerun here.

## 2. Multilingual concept retrieval

**Question:** Do aligned Tunisian/Arabizi/French phrases or multilingual MiniLM recover the intended concept?

**Historical protocol:** 44 known development examples and 10 later spelling/idiom stress examples. This measures concept sets, not clinical spans or context. Known examples informed dictionary development.

[Experiment code](research/experiment.py) · [Full saved results](artifacts/multilingual/comparison.json)

In [2]:
record = read_json(RESEARCH / 'comparison.json')
rows = []
for approach, result in record['approaches'].items():
    parts = result['by_partition']
    rows.append({
        'Approach': approach,
        'Development matches': f"{parts['development']['correct']}/44",
        'Stress matches': f"{parts['additional_stress']['correct']}/10",
        'Mean CPU ms': result['mean_extraction_ms'],
    })
display(pd.DataFrame(rows).set_index('Approach'))

,Development matches,Stress matches,Mean CPU ms
Approach,,,
existing_english,14/44,1/10,0.1827
aligned_clinical_phrases,44/44,3/10,0.3508
multilingual_encoder,23/44,4/10,2.2827


**Takeaway:** Phrases matched 44/44 known development examples but only 3/10 stress cases. MiniLM recovered 4/10 stress cases; sentence retrieval did not establish reliable evidence, assertion, time or experiencer.

## 3. Full-fact extraction trials

**Question:** Does a transformer improve the complete extracted fact, including spans and clinical context?

**Historical protocol:** 12 synthetic development cases and 24 synthetic challenge cases with 31 entities. Complete-fact scoring includes canonical concept, assertion, timing, experiencer and specified attributes. These inspected fixtures await independent native-speaker and clinician review.

[Experiment code](research/hybrid_experiment.py) · [References](artifacts/multilingual/hybrid_reference.jsonl) · [Full results and failures](artifacts/multilingual/hybrid_final.json)

In [3]:
record = read_json(RESEARCH / 'hybrid_final.json')
approaches = {
    'phase1': 'Earlier dictionaries',
    'improved_dictionary': 'Improved dictionaries (default)',
    'minilm_span_context': 'MiniLM span retrieval',
    'gliner_contextual_native': 'GLiNER INT8',
    'gliner_fp32_native': 'GLiNER FP32',
    'gliner_fp32_conservative_hybrid': 'Dictionaries + uncertain FP32 proposals',
}
rows = []
for key, label in approaches.items():
    result = record['approaches'][key]
    rows.append({
        'Approach': label,
        'Entity F1': result['exact_span_entities']['f1'],
        'Complete-fact F1': result['complete_facts_overlap']['f1'],
        'Critical recall': result['critical_symptom_recall'],
        'Unsupported span rate': result['unsupported_fact_rate'],
        'Median CPU ms': result['latency_ms']['median'],
    })
display(pd.DataFrame(rows).set_index('Approach'))

,Entity F1,Complete-fact F1,Critical recall,Unsupported span rate,Median CPU ms
Approach,,,,,
Earlier dictionaries,0.9153,0.8136,0.3333,0.0000,0.4546
Improved dictionaries (default),0.9333,0.8667,0.3333,0.0000,0.4531
MiniLM span retrieval,0.2545,0.2182,0.3333,0.7595,32.1390
GLiNER INT8,0.0000,0.0000,0.0000,0.0000,47.6113
GLiNER FP32,0.4878,0.4390,0.0000,0.0000,394.8207
Dictionaries + uncertain FP32 proposals,0.9333,0.8667,0.3333,0.0000,380.8970


**Takeaway:** Improved dictionaries reached complete-fact F1 0.867 on this small challenge. No evaluated transformer improved it. Critical symptom recall stayed at 1/3. The INT8 export failed its smoke control; this is a finding about the tested export/configuration. Optional FP32 proposals stay uncertain until confirmed.

## 4. Tokenizer investigation

**Question:** Can candidate tokenizers encode the mixed-language strings without unknown tokens?

**Historical protocol:** Nine shared examples; fragmentation is subwords per space-separated word.

[Raw token sequences, revisions and licenses](artifacts/multilingual/tokenization.json)

In [4]:
records = read_json(RESEARCH / 'tokenization.json')
rows = []
for record in records:
    samples = record['samples']
    rows.append({
        'Tokenizer': record['model'],
        'Mean subwords/word': np.mean([s['subwords_per_space_word'] for s in samples]),
        'Unknown tokens': sum(s['unknown_tokens'] for s in samples),
    })
display(pd.DataFrame(rows).set_index('Tokenizer'))

,Mean subwords/word,Unknown tokens
Tokenizer,,
tunis-ai/TunBERT,2.5132,0
FacebookAI/xlm-roberta-base,2.0161,0
Vionex-digital/vioBERT-v3,2.3712,0
onnx-community/gliner_multi-v2.1,2.1365,0


**Takeaway:** The tested examples produced no unknown tokens. That establishes encoding coverage for those examples, not clinical understanding.

## 5. Selected extractor: live examples

The default aligns multilingual phrases to original evidence, applies bounded clinical context, and builds structured facts. Optional semantic proposals require confirmation before entering ML features.

| Source | Responsibility |
|---|---|
| [multilingual_nlp.py](src/multilingual_nlp.py) | Aligned phrases and optional semantic proposals |
| [clinical_state.py](src/clinical_state.py) | Assertion, timing, experiencer, values and state |
| [adaptive_questions.py](src/adaptive_questions.py) | Missing-slot questions and linked answers |
| [clinical_rules.py](src/clinical_rules.py) | Independent evidence-backed screens |
| [triage_pipeline.py](src/triage_pipeline.py) | Public extraction and turn API |

**Current run:** These facts come from the bundled extractor.

In [5]:
examples = [
    '3andi wja3 fi sadri',
    'صدري يوجعني',
    'My father has asthma. I have dizziness now.',
    'I have no chest pain, but I have a cough.',
    'Je prends du paracétamol 500 mg.',
]
tables = []
for text in examples:
    table = facts_frame(extract_info(text)['facts'])
    table.insert(0, 'Input', text)
    tables.append(table)
display(pd.concat(tables, ignore_index=True))

,Input,text,kind,name,status,temporality,experiencer,severity
0,3andi wja3 fi sadri,wja3 fi sadri,anatomy,chest,confirmed,current,patient,None
1,3andi wja3 fi sadri,wja3 fi sadri,symptom,chest,confirmed,current,patient,None
2,صدري يوجعني,صدري يوجعني,anatomy,chest,confirmed,current,patient,None
3,صدري يوجعني,صدري يوجعني,symptom,chest,confirmed,current,patient,None
4,My father has asthma. I have dizziness now.,asthma,condition,asthma,confirmed,current,other:father,None
5,My father has asthma. I have dizziness now.,dizziness,symptom,dizziness,confirmed,current,patient,None
6,"I have no chest pain, but I have a cough.",chest,anatomy,chest,negated,current,patient,None
7,"I have no chest pain, but I have a cough.",chest pain,symptom,chest,negated,current,patient,None
8,"I have no chest pain, but I have a cough.",cough,symptom,cough,confirmed,current,patient,None
9,Je prends du paracétamol 500 mg.,paracétamol,medication,acetaminophen,confirmed,current,patient,None


## 6. Current reference checks

**Current run:** Score the existing 36 English and 15 multilingual inspected development references. These informed development and are not an independent test. Detailed outputs go to the ignored `.notebook_runtime/fresh_checks/` folder.

In [6]:
from pipeline_validation import evaluate_nlp
references = {
    'English': ARTIFACTS / 'nlp_evaluation/manual_reference.jsonl',
    'Multilingual': RESEARCH / 'reference.jsonl',
}
rows = []
for label, reference in references.items():
    result = evaluate_nlp(reference, output_dir=WORK / 'fresh_checks' / label)
    rows.append({'Reference': label, 'Cases': result['reference_cases'],
                 **result['entity_exact_span_and_name']})
display(pd.DataFrame(rows).set_index('Reference'))

NLP entity identity: {'tp': 49, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


       attribute  reference_count  entity_found_count  correct_count  end_to_end_accuracy  accuracy_on_found_entities
      dose_value                2                   2              2                  1.0                         1.0
duration_seconds                2                   2              2                  1.0                         1.0
     experiencer               49                  49             49                  1.0                         1.0
 medication_role                5                   5              5                  1.0                         1.0
           onset                1                   1              1                  1.0                         1.0
        severity                6                   6              6                  1.0                         1.0
          status               49                  49             49                  1.0                         1.0
     temporality               49                  49   

NLP entity identity: {'tp': 15, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


       attribute  reference_count  entity_found_count  correct_count  end_to_end_accuracy  accuracy_on_found_entities
      dose_value                1                   1              1                  1.0                         1.0
duration_seconds                1                   1              1                  1.0                         1.0
     experiencer               15                  15             15                  1.0                         1.0
 medication_role                2                   2              2                  1.0                         1.0
           onset                1                   1              1                  1.0                         1.0
        severity                1                   1              1                  1.0                         1.0
          status               15                  15             15                  1.0                         1.0
     temporality               15                  15   

,Cases,tp,fp,fn,precision,recall,f1
Reference,,,,,,,
English,36,48,1,1,0.9796,0.9796,0.9796
Multilingual,15,15,0,0,1.0000,1.0000,1.0000


## 7. Multi-turn behavior

**Current run:** Start an Arabizi complaint and answer the generated question using its ID. Known critical rules interrupt questioning and can defer independent ML.

In [7]:
first = process_turn(None, '3andi wja3 fi sadri', model_version='experimental_v3')
question = first['next_question']
display(pd.DataFrame([{'Patient': '3andi wja3 fi sadri', 'Next question': question}]))
if question:
    final = process_turn(first['patient_state'], 'Ey, barcha', question['id'], model_version='experimental_v3')
    display(facts_frame(final['patient_state']['facts']))
    display(pd.DataFrame([{
        'Critical flag': final['rules']['critical_flags_detected'],
        'ML status': final['ml']['status'],
        'Next question': final['next_question'],
    }]))

C:\Users\moura\Desktop\hackcure\hackcure-nlp-urgency\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Patient,Next question
0,3andi wja3 fi sadri,"{'field': 'symptom.breath', 'priority': 100, 'reason': 'Relev..."


,text,kind,name,status,temporality,experiencer,severity
0,wja3 fi sadri,anatomy,chest,confirmed,current,patient,None
1,wja3 fi sadri,symptom,chest,confirmed,current,patient,None
2,"Ey, barcha",symptom,breath,confirmed,current,patient,"{'text': 'barcha', 'value': 'intense', 'start': 4, 'end': 10,..."


,Critical flag,ML status,Next question
0,True,deferred_critical_fast_path,None


**Historical integration traces:** This summary reads saved English, Arabizi and Arabic conversations. [Full messages, facts and evidence](artifacts/multilingual/finalization_conversations.json).

In [8]:
traces = read_json(RESEARCH / 'finalization_conversations.json')
rows = []
for language, conversation in traces.items():
    last = conversation['trace'][-1]
    rows.append({
        'Language': language,
        'Turns': len(conversation['messages']),
        'Critical interruption': last['rules']['critical_flags_detected'],
        'Independent ML score': conversation['independent_final_ml'].get('model_score_emergency'),
    })
display(pd.DataFrame(rows).set_index('Language'))

,Turns,Critical interruption,Independent ML score
Language,,,
en,7,True,0.9996
tn_arabizi,7,True,0.9996
tn_arabic,7,True,0.9996


## 8. Conclusions and next work

- Selected default: aligned dictionaries, clinical context and exact evidence.
- Two unfamiliar respiratory descriptions were missed in the full-fact challenge; critical recall was 1/3.
- MiniLM and GLiNER did not improve complete-fact extraction on this reference. No transformer is enabled by default.
- The ML context-feature regression remains unresolved. Compatibility does not establish clinical generalization.
- Further validation needs independently annotated Tunisian clinical spans and context, with native-speaker and clinician review.

[Detailed research notes and primary sources](artifacts/multilingual/README.md) · [Historical finalization notes](artifacts/multilingual/finalization_summary.json) · [Research rerun requirements](research/README.md)